In [2]:
import json

# Load sample_candidates.json (only 50 candidates, small file)
with open("../sample_candidates.json") as f:
    candidates = json.load(f)

print(f"Total candidates: {len(candidates)}")

# Question 1: How many are open to work?
open_to_work = [c for c in candidates if c["redrob_signals"]["open_to_work_flag"]]
print(f"Open to work: {len(open_to_work)}")

# Question 2: Average recruiter response rate
avg_response = sum(c["redrob_signals"]["recruiter_response_rate"] for c in candidates) / len(candidates)
print(f"Avg recruiter response rate: {avg_response:.2f}")

# Question 3: How many have NO GitHub linked?
no_github = [c for c in candidates if c["redrob_signals"]["github_activity_score"] == -1]
print(f"No GitHub: {len(no_github)}")

# Question 4: Most common skills
from collections import Counter
all_skills = []
for c in candidates:
    for s in c["skills"]:
        all_skills.append(s["name"])
print("\nTop 15 skills:")
for skill, count in Counter(all_skills).most_common(15):
    print(f"  {skill}: {count}")

# Question 5: Skill claim vs assessment score mismatch
print("\nSelf-reported vs assessed skill mismatches:")
for c in candidates:
    assessed = c["redrob_signals"]["skill_assessment_scores"]
    for skill in c["skills"]:
        if skill["name"] in assessed:
            if skill["proficiency"] in ["advanced", "expert"] and assessed[skill["name"]] < 50:
                print(f'  {c["candidate_id"]}: claims {skill["proficiency"]} {skill["name"]} but scored {assessed[skill["name"]]:.1f}')

Total candidates: 50
Open to work: 16
Avg recruiter response rate: 0.44
No GitHub: 33

Top 15 skills:
  AWS: 10
  Data Pipelines: 10
  Node.js: 9
  Sales: 9
  Figma: 9
  Tailwind: 8
  GCP: 8
  Content Writing: 8
  GraphQL: 8
  gRPC: 8
  Terraform: 8
  Scrum: 8
  Java: 8
  Spring Boot: 8
  Hadoop: 8

Self-reported vs assessed skill mismatches:
  CAND_0000001: claims advanced NLP but scored 38.8
  CAND_0000001: claims advanced Fine-tuning LLMs but scored 41.6
  CAND_0000011: claims advanced Recommendation Systems but scored 29.8
  CAND_0000025: claims advanced LangChain but scored 40.0
  CAND_0000027: claims advanced Data Science but scored 35.1


In [5]:
# Check honeypot-style contradictions
print("Potential honeypot signals:")
for c in candidates:
    sig = c["redrob_signals"]
    cid = c["candidate_id"]
    flags = []
    
    # Contradiction 1: open to work but very inactive
    from datetime import datetime, date
    last_active = datetime.strptime(sig["last_active_date"], "%Y-%m-%d").date()
    days_inactive = (date.today() - last_active).days
    if sig["open_to_work_flag"] and days_inactive > 180:
        flags.append(f"open_to_work but inactive {days_inactive} days")
    
    # Contradiction 2: high endorsements but zero connections
    if sig["endorsements_received"] > 50 and sig["connection_count"] < 10:
        flags.append(f"endorsements={sig['endorsements_received']} but connections={sig['connection_count']}")
    
    # Contradiction 3: perfect offer acceptance but zero interview completion
    if sig["offer_acceptance_rate"] == 1.0 and sig["interview_completion_rate"] < 0.1:
        flags.append("offer_acceptance=1.0 but never completes interviews")
    
    # Contradiction 4: 100% profile complete but unverified
    if sig["profile_completeness_score"] == 100 and not sig["verified_email"]:
        flags.append("completeness=100 but email unverified")

    if flags:
        print(f"\n  {cid}:")
        for f in flags:
            print(f"    ⚠ {f}")

Potential honeypot signals:

  CAND_0000002:
    ⚠ open_to_work but inactive 212 days

  CAND_0000005:
    ⚠ open_to_work but inactive 254 days

  CAND_0000036:
    ⚠ open_to_work but inactive 182 days


In [6]:
def is_hard_disqualified(candidate):
    companies = [r["company"] for r in candidate["career_history"]]
    consulting = ["TCS","Infosys","Wipro","Accenture","Cognizant","Capgemini"]
    
    # All consulting, no product company
    all_consulting = all(any(c in comp for c in consulting) for comp in companies)
    if all_consulting:
        return True, "pure consulting background"
    
    # Inactive too long
    from datetime import datetime, date
    last_active = datetime.strptime(candidate["redrob_signals"]["last_active_date"], "%Y-%m-%d").date()
    if (date.today() - last_active).days > 180:
        return True, "inactive 6+ months"
    
    return False, None

In [7]:
disqualified = []
passed = []

for c in candidates:
    failed, reason = is_hard_disqualified(c)
    if failed:
        disqualified.append((c["candidate_id"], reason))
    else:
        passed.append(c)

print(f"Total: {len(candidates)}")
print(f"Disqualified: {len(disqualified)}")
print(f"Passed: {len(passed)}")
print("\nDisqualified list:")
for cid, reason in disqualified:
    print(f"  {cid}: {reason}")

Total: 50
Disqualified: 20
Passed: 30

Disqualified list:
  CAND_0000002: inactive 6+ months
  CAND_0000003: pure consulting background
  CAND_0000005: inactive 6+ months
  CAND_0000008: pure consulting background
  CAND_0000012: inactive 6+ months
  CAND_0000017: inactive 6+ months
  CAND_0000020: inactive 6+ months
  CAND_0000021: inactive 6+ months
  CAND_0000024: pure consulting background
  CAND_0000026: inactive 6+ months
  CAND_0000027: pure consulting background
  CAND_0000028: pure consulting background
  CAND_0000029: inactive 6+ months
  CAND_0000030: inactive 6+ months
  CAND_0000036: inactive 6+ months
  CAND_0000037: inactive 6+ months
  CAND_0000042: inactive 6+ months
  CAND_0000044: inactive 6+ months
  CAND_0000047: pure consulting background
  CAND_0000050: inactive 6+ months


In [8]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
test = model.encode(["test sentence"])
print("Model loaded, shape:", test.shape)

C:\Users\KIIT0001\AppData\Roaming\Python\Python313\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded, shape: (1, 384)


In [9]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from datetime import datetime, date

model = SentenceTransformer("all-MiniLM-L6-v2")

# What the JD actually needs — in plain language
JD_TEXT = """
Senior AI Engineer with production experience in embeddings-based retrieval systems,
vector databases, hybrid search, ranking evaluation metrics like NDCG and MRR,
strong Python, applied ML at product companies, LLM fine-tuning with LoRA or QLoRA,
building recommendation and ranking systems, shipping end-to-end ML systems to real users,
evaluation frameworks for ranking, A/B testing, retrieval quality measurement.
Experience at product companies not consulting firms. 5 to 9 years experience.
"""

jd_embedding = model.encode([JD_TEXT])

def score_candidate(candidate):
    sig = candidate["redrob_signals"]
    
    # --- 1. Semantic skill match (35%) ---
    career_text = " ".join([r["description"] for r in candidate["career_history"]])
    skills_text = " ".join([s["name"] for s in candidate["skills"]])
    full_text = career_text + " " + skills_text
    
    candidate_embedding = model.encode([full_text])
    semantic_score = cosine_similarity(jd_embedding, candidate_embedding)[0][0]
    
    # If semantically too far from JD, not relevant at all
    if semantic_score < 0.28:
        return 0.0
    
    # ML keyword boost — reward candidates whose career actually mentions ML work
    ML_KEYWORDS = ["retrieval", "ranking", "recommendation", "embeddings",
                   "vector", "nlp", "fine-tun", "transformer", "search"]
    career_lower = career_text.lower()
    ml_hits = sum(1 for kw in ML_KEYWORDS if kw in career_lower)
    ml_boost = min(ml_hits * 0.03, 0.15)
    semantic_score = min(semantic_score + ml_boost, 1.0)

    # --- 2. Title relevance penalty ---
    RELEVANT_TITLES = ["engineer", "developer", "scientist", "ml", "ai", "data",
                       "backend", "research", "nlp", "architect"]
    IRRELEVANT_TITLES = ["graphic", "civil", "mechanical", "accountant", "marketing",
                         "operations", "mobile", "project manager", "customer support",
                         "business analyst", "qa", "frontend", "java", ".net"]
    
    title = candidate["profile"]["current_title"].lower()
    title_penalty = 1.0
    if any(t in title for t in IRRELEVANT_TITLES):
        title_penalty = 0.25
    elif any(t in title for t in RELEVANT_TITLES):
        title_penalty = 1.0

    # --- 3. Experience quality (25%) ---
    consulting = ["TCS","Infosys","Wipro","Accenture","Cognizant","Capgemini"]
    exp_score = 0
    
    yoe = candidate["profile"]["years_of_experience"]
    if 5 <= yoe <= 9:
        exp_score += 0.4
    elif 4 <= yoe <= 11:
        exp_score += 0.2
    
    for role in candidate["career_history"]:
        is_consulting = any(c in role["company"] for c in consulting)
        if not is_consulting:
            exp_score += 0.1
    exp_score = min(exp_score, 1.0)

    # --- 4. Availability / behavioral (20%) ---
    avail_score = 0
    if sig["open_to_work_flag"]:
        avail_score += 0.3
    
    last_active = datetime.strptime(sig["last_active_date"], "%Y-%m-%d").date()
    days_inactive = (date.today() - last_active).days
    if days_inactive < 30:
        avail_score += 0.3
    elif days_inactive < 90:
        avail_score += 0.2
    
    avail_score += sig["recruiter_response_rate"] * 0.2
    if sig["notice_period_days"] <= 30:
        avail_score += 0.2
    elif sig["notice_period_days"] <= 60:
        avail_score += 0.1
    avail_score = min(avail_score, 1.0)

    # --- 5. Verified signals (12%) ---
    verify_score = 0
    assessed = sig["skill_assessment_scores"]
    if assessed:
        avg_assessment = sum(assessed.values()) / len(assessed)
        verify_score += (avg_assessment / 100) * 0.5
    if sig["github_activity_score"] > 0:
        verify_score += (sig["github_activity_score"] / 100) * 0.3
    if sig["verified_email"] and sig["verified_phone"]:
        verify_score += 0.2

    # --- 6. Nice to haves (8%) ---
    bonus_score = 0
    nice_skills = ["lora", "qlora", "peft", "fine-tuning", "open-source",
                   "learning to rank", "xgboost", "hr-tech", "recruiting"]
    candidate_skills_lower = [s["name"].lower() for s in candidate["skills"]]
    for skill in nice_skills:
        if any(skill in s for s in candidate_skills_lower):
            bonus_score += 0.15
    bonus_score = min(bonus_score, 1.0)

    # --- 7. Job hopper penalty ---
    short_stints = [r for r in candidate["career_history"]
                    if r["duration_months"] < 12 and not r["is_current"]]
    hopper_penalty = 1.0
    if len(short_stints) >= 2:
        hopper_penalty = 0.7
    if len(short_stints) >= 3:
        hopper_penalty = 0.5

    # --- Final weighted score ---
    final = (
        semantic_score * 0.35 +
        exp_score      * 0.25 +
        avail_score    * 0.20 +
        verify_score   * 0.12 +
        bonus_score    * 0.08
    )
    
    return round(final * title_penalty * hopper_penalty, 4)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
results = []
for c in passed:
    score = score_candidate(c)
    results.append({
        "candidate_id": c["candidate_id"],
        "name": c["profile"]["anonymized_name"],
        "title": c["profile"]["current_title"],
        "yoe": c["profile"]["years_of_experience"],
        "score": score
    })

results.sort(key=lambda x: x["score"], reverse=True)

print("RANKED CANDIDATES")
print("-" * 60)
for i, r in enumerate(results, 1):
    print(f"{i:2}. {r['candidate_id']} | {r['score']:.4f} | {r['yoe']}yrs | {r['title']}")

RANKED CANDIDATES
------------------------------------------------------------
 1. CAND_0000031 | 0.5690 | 6.0yrs | Recommendation Systems Engineer
 2. CAND_0000010 | 0.3833 | 4.6yrs | Data Engineer
 3. CAND_0000043 | 0.3551 | 8.3yrs | Cloud Engineer
 4. CAND_0000023 | 0.3304 | 3.7yrs | Software Engineer
 5. CAND_0000035 | 0.2560 | 4.3yrs | Full Stack Developer
 6. CAND_0000033 | 0.1123 | 8.6yrs | Graphic Designer
 7. CAND_0000014 | 0.1118 | 8.4yrs | Frontend Engineer
 8. CAND_0000045 | 0.1101 | 12.2yrs | Project Manager
 9. CAND_0000038 | 0.1100 | 6.7yrs | Java Developer
10. CAND_0000019 | 0.1079 | 6.5yrs | Project Manager
11. CAND_0000007 | 0.1030 | 5.5yrs | Civil Engineer
12. CAND_0000016 | 0.1022 | 5.3yrs | Accountant
13. CAND_0000046 | 0.0946 | 7.8yrs | Mechanical Engineer
14. CAND_0000041 | 0.0907 | 13.7yrs | Operations Manager
15. CAND_0000006 | 0.0878 | 6.0yrs | Business Analyst
16. CAND_0000009 | 0.0796 | 11.0yrs | Mechanical Engineer
17. CAND_0000004 | 0.0791 | 3.8yrs | Marke

In [11]:
# Debug a specific candidate
c = next(c for c in passed if c["candidate_id"] == "CAND_0000015")
career_text = " ".join([r["description"] for r in c["career_history"]])
skills_text = " ".join([s["name"] for s in c["skills"]])
full_text = career_text + " " + skills_text

emb = model.encode([full_text])
score = cosine_similarity(jd_embedding, emb)[0][0]
print(f"Semantic score: {score:.4f}")
print(f"Title: {c['profile']['current_title']}")
print(f"\nCareer description preview:")
print(career_text[:300])

Semantic score: 0.2358
Title: Software Engineer

Career description preview:
Cloud infrastructure and DevOps work at an enterprise SaaS company. Owned the AWS account architecture (VPC, IAM, networking), the Terraform modules for our service deployments, and the Kubernetes cluster operations. Designed the CI/CD pipelines (GitLab CI + ArgoCD) and the monitoring stack (Prometh


In [12]:
c = next(c for c in passed if c["candidate_id"] == "CAND_0000031")
for role in c["career_history"]:
    print(f"{role['title']} @ {role['company']} — {role['duration_months']} months | current: {role['is_current']}")

Recommendation Systems Engineer @ Swiggy — 14 months | current: True
Search Engineer @ Mad Street Den — 16 months | current: False
NLP Engineer @ Uber — 27 months | current: False
Applied ML Engineer @ Zomato — 13 months | current: False


In [13]:
c = next(c for c in passed if c["candidate_id"] == "CAND_0000001")
career_text = " ".join([r["description"] for r in c["career_history"]])
emb = model.encode([career_text])
sem = cosine_similarity(jd_embedding, emb)[0][0]
ML_KEYWORDS = ["retrieval", "ranking", "recommendation", "embeddings",
               "vector", "nlp", "fine-tun", "transformer", "search"]
ml_hits = sum(1 for kw in ML_KEYWORDS if kw in career_text.lower())
print(f"Semantic: {sem:.4f} | ML hits: {ml_hits}")

Semantic: 0.2355 | ML hits: 0


In [14]:
import gzip, json

all_candidates = []
with open("../candidates.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            all_candidates.append(json.loads(line))

print(f"Loaded: {len(all_candidates)} candidates")

Loaded: 100000 candidates


In [15]:
# Step 1: Hard disqualify first, encode ONLY survivors
# This cuts 100k down to maybe 40-50k before any encoding

def is_hard_disqualified(candidate):
    consulting = ["TCS","Infosys","Wipro","Accenture","Cognizant","Capgemini"]
    companies = [r["company"] for r in candidate["career_history"]]
    all_consulting = all(any(c in comp for c in consulting) for comp in companies)
    if all_consulting:
        return True
    
    last_active = datetime.strptime(
        candidate["redrob_signals"]["last_active_date"], "%Y-%m-%d").date()
    if (date.today() - last_active).days > 180:
        return True
    
    return False

# Filter first
survivors = [c for c in all_candidates if not is_hard_disqualified(c)]
print(f"Before: {len(all_candidates)} | After filter: {len(survivors)}")

Before: 100000 | After filter: 70623


In [16]:
JD_TEXT = """
Senior AI Engineer with production experience in embeddings-based retrieval systems,
vector databases, hybrid search, ranking evaluation metrics like NDCG and MRR,
strong Python, applied ML at product companies, LLM fine-tuning with LoRA or QLoRA,
building recommendation and ranking systems, shipping end-to-end ML systems to real users,
evaluation frameworks for ranking, A/B testing, retrieval quality measurement.
Experience at product companies not consulting firms. 5 to 9 years experience.
"""

jd_embedding = model.encode([JD_TEXT])
print("JD embedded, shape:", jd_embedding.shape)

JD embedded, shape: (1, 384)


In [17]:
def score_candidate(candidate):
    sig = candidate["redrob_signals"]
    
    career_text = " ".join([r["description"] for r in candidate["career_history"]])
    skills_text = " ".join([s["name"] for s in candidate["skills"]])
    full_text = career_text + " " + skills_text
    
    candidate_embedding = model.encode([full_text])
    semantic_score = cosine_similarity(jd_embedding, candidate_embedding)[0][0]
    
    if semantic_score < 0.28:
        return 0.0
    
    ML_KEYWORDS = ["retrieval", "ranking", "recommendation", "embeddings",
                   "vector", "nlp", "fine-tun", "transformer", "search"]
    career_lower = career_text.lower()
    ml_hits = sum(1 for kw in ML_KEYWORDS if kw in career_lower)
    ml_boost = min(ml_hits * 0.03, 0.15)
    semantic_score = min(semantic_score + ml_boost, 1.0)

    RELEVANT_TITLES = ["engineer", "developer", "scientist", "ml", "ai", "data",
                       "backend", "research", "nlp", "architect"]
    IRRELEVANT_TITLES = ["graphic", "civil", "mechanical", "accountant", "marketing",
                         "operations", "mobile", "project manager", "customer support",
                         "business analyst", "qa", "frontend", "java", ".net"]
    title = candidate["profile"]["current_title"].lower()
    title_penalty = 1.0
    if any(t in title for t in IRRELEVANT_TITLES):
        title_penalty = 0.25
    
    consulting = ["TCS","Infosys","Wipro","Accenture","Cognizant","Capgemini"]
    exp_score = 0
    yoe = candidate["profile"]["years_of_experience"]
    if 5 <= yoe <= 9:
        exp_score += 0.4
    elif 4 <= yoe <= 11:
        exp_score += 0.2
    for role in candidate["career_history"]:
        if not any(c in role["company"] for c in consulting):
            exp_score += 0.1
    exp_score = min(exp_score, 1.0)

    avail_score = 0
    if sig["open_to_work_flag"]:
        avail_score += 0.3
    last_active = datetime.strptime(sig["last_active_date"], "%Y-%m-%d").date()
    days_inactive = (date.today() - last_active).days
    if days_inactive < 30:
        avail_score += 0.3
    elif days_inactive < 90:
        avail_score += 0.2
    avail_score += sig["recruiter_response_rate"] * 0.2
    if sig["notice_period_days"] <= 30:
        avail_score += 0.2
    elif sig["notice_period_days"] <= 60:
        avail_score += 0.1
    avail_score = min(avail_score, 1.0)

    verify_score = 0
    assessed = sig["skill_assessment_scores"]
    if assessed:
        avg_assessment = sum(assessed.values()) / len(assessed)
        verify_score += (avg_assessment / 100) * 0.5
    if sig["github_activity_score"] > 0:
        verify_score += (sig["github_activity_score"] / 100) * 0.3
    if sig["verified_email"] and sig["verified_phone"]:
        verify_score += 0.2

    bonus_score = 0
    nice_skills = ["lora", "qlora", "peft", "fine-tuning", "open-source",
                   "learning to rank", "xgboost", "hr-tech", "recruiting"]
    candidate_skills_lower = [s["name"].lower() for s in candidate["skills"]]
    for skill in nice_skills:
        if any(skill in s for s in candidate_skills_lower):
            bonus_score += 0.15
    bonus_score = min(bonus_score, 1.0)

    short_stints = [r for r in candidate["career_history"]
                    if r["duration_months"] < 12 and not r["is_current"]]
    hopper_penalty = 1.0
    if len(short_stints) >= 2:
        hopper_penalty = 0.7
    if len(short_stints) >= 3:
        hopper_penalty = 0.5

    final = (
        semantic_score * 0.35 +
        exp_score      * 0.25 +
        avail_score    * 0.20 +
        verify_score   * 0.12 +
        bonus_score    * 0.08
    )
    
    return round(final * title_penalty * hopper_penalty, 4)

print("score_candidate() ready")

score_candidate() ready


In [20]:
import heapq, os
TOP_N = 500
top_candidates = []
processed = 0
skipped = 0
consulting = ["TCS","Infosys","Wipro","Accenture","Cognizant","Capgemini"]

os.makedirs("progress", exist_ok=True)

with open("../candidates.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        
        c = json.loads(line)
        
        companies = [r["company"] for r in c["career_history"]]
        all_consulting = all(any(con in comp for con in consulting) for comp in companies)
        last_active = datetime.strptime(c["redrob_signals"]["last_active_date"], "%Y-%m-%d").date()
        inactive = (date.today() - last_active).days > 180
        
        if all_consulting or inactive:
            skipped += 1
            del c
            processed += 1
            continue
        
        score = score_candidate(c)
        
        if score > 0:
            if len(top_candidates) < TOP_N:
                heapq.heappush(top_candidates, (score, c["candidate_id"], c))
            elif score > top_candidates[0][0]:
                heapq.heapreplace(top_candidates, (score, c["candidate_id"], c))
        
        del c
        processed += 1
        
        if processed % 1000 == 0:
            checkpoint = sorted(top_candidates, key=lambda x: x[0], reverse=True)
            checkpoint_out = [{"candidate_id": cid, "score": float(s),
                               "title": cd["profile"]["current_title"],
                               "yoe": cd["profile"]["years_of_experience"],
                               "name": cd["profile"]["anonymized_name"]}
                              for s, cid, cd in checkpoint]
            with open("progress/checkpoint.json", "w") as out:
                json.dump(checkpoint_out, out)
        
        if processed % 5000 == 0:
            print(f"{processed}/100000 | skipped: {skipped} | top score: {top_candidates[0][0]:.4f}")

final_ranking = sorted(top_candidates, key=lambda x: x[0], reverse=True)
final_out = [{"candidate_id": cid, "score": float(s),
              "title": cd["profile"]["current_title"],
              "yoe": cd["profile"]["years_of_experience"],
              "name": cd["profile"]["anonymized_name"]}
             for s, cid, cd in final_ranking]

with open("top500_final.json", "w") as f:
    json.dump(final_out, f, indent=2)

print(f"Saved to top500_final.json!")
print("\nTop 10:")
for i, r in enumerate(final_out[:10], 1):
    print(f"{i:2}. {r['candidate_id']} | {r['score']:.4f} | {r['yoe']}yrs | {r['title']}")

5000/100000 | skipped: 1407 | top score: 0.3607
10000/100000 | skipped: 2891 | top score: 0.4121
15000/100000 | skipped: 4385 | top score: 0.4347
25000/100000 | skipped: 7295 | top score: 0.4601
30000/100000 | skipped: 8750 | top score: 0.4695
35000/100000 | skipped: 10310 | top score: 0.4762
55000/100000 | skipped: 16155 | top score: 0.4957
75000/100000 | skipped: 21980 | top score: 0.5094
80000/100000 | skipped: 23438 | top score: 0.5137
85000/100000 | skipped: 24891 | top score: 0.5167
90000/100000 | skipped: 26356 | top score: 0.5188
95000/100000 | skipped: 27847 | top score: 0.5208
Saved to top500_final.json!

Top 10:
 1. CAND_0079387 | 0.7184 | 6.9yrs | AI Engineer
 2. CAND_0018499 | 0.7177 | 7.2yrs | Senior Machine Learning Engineer
 3. CAND_0064326 | 0.7054 | 7.6yrs | Search Engineer
 4. CAND_0041610 | 0.7047 | 6.7yrs | Recommendation Systems Engineer
 5. CAND_0027691 | 0.6973 | 6.5yrs | NLP Engineer
 6. CAND_0041669 | 0.6947 | 8.0yrs | Recommendation Systems Engineer
 7. CAND_

In [25]:
import csv, json

# Load your saved results
with open("top500_final.json") as f:
    top500 = json.load(f)

# Take only top 100 for submission
top100 = top500[:100]

# Sort by score descending, then candidate_id ascending for ties
top100_sorted = sorted(top100, key=lambda x: (-x["score"], x["candidate_id"]))

# Write CSV in required format
with open("submission.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["candidate_id", "rank", "score", "reasoning"])
    
    for i, r in enumerate(top100_sorted, 1):
        reasoning = (
            f"{r['title']} with {r['yoe']} years of experience. "
            f"Strong semantic match to Senior AI Engineer role based on career history, "
            f"skills alignment, and behavioral signals. Score: {r['score']:.4f}."
        )
        writer.writerow([r["candidate_id"], i, round(r["score"], 4), reasoning])

print("submission.csv saved!")
print(f"Rows: {sum(1 for _ in open('submission.csv')) - 1}")

submission.csv saved!
Rows: 100


In [27]:
with open("top500_final.json") as f:
    top500 = json.load(f)

# We need full candidate data for reasoning
# Load a lookup from the original file
import json
candidate_lookup = {}
with open("../candidates.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            c = json.loads(line)
            if c["candidate_id"] in [r["candidate_id"] for r in top500[:100]]:
                candidate_lookup[c["candidate_id"]] = c

print(f"Loaded {len(candidate_lookup)} candidate profiles")

Loaded 100 candidate profiles


In [28]:
def generate_reasoning(c, rank, score):
    sig = c["redrob_signals"]
    profile = c["profile"]
    
    # Key facts
    title = profile["current_title"]
    yoe = profile["years_of_experience"]
    company = profile["current_company"]
    top_skills = [s["name"] for s in sorted(c["skills"], 
                  key=lambda x: x["endorsements"], reverse=True)[:3]]
    notice = sig["notice_period_days"]
    response_rate = sig["recruiter_response_rate"]
    assessed = sig["skill_assessment_scores"]
    
    # Build specific reasoning
    skills_str = ", ".join(top_skills) if top_skills else "varied skills"
    
    # Positive part
    reasoning = f"{title} with {yoe} years experience at {company}"
    
    if assessed:
        best_skill = max(assessed, key=assessed.get)
        best_score = assessed[best_skill]
        reasoning += f"; verified {best_skill} assessment score {best_score:.0f}/100"
    
    # JD connection
    career_text = " ".join([r["description"] for r in c["career_history"]]).lower()
    ml_signals = []
    if "retrieval" in career_text: ml_signals.append("retrieval systems")
    if "ranking" in career_text: ml_signals.append("ranking")
    if "recommendation" in career_text: ml_signals.append("recommendation systems")
    if "embedding" in career_text: ml_signals.append("embeddings")
    if "nlp" in career_text: ml_signals.append("NLP")
    
    if ml_signals:
        reasoning += f"; career history shows hands-on {', '.join(ml_signals[:2])} work"
    
    # Honest concerns
    concerns = []
    if notice > 60:
        concerns.append(f"notice period is {notice} days")
    if response_rate < 0.3:
        concerns.append(f"low recruiter response rate ({response_rate:.0f}%)")
    if rank > 50:
        concerns.append("adjacent skills only — borderline fit")
    
    if concerns:
        reasoning += f". Concern: {'; '.join(concerns)}."
    else:
        reasoning += "."
    
    return reasoning

# Test on top 5
top100_sorted = sorted(top500[:100], key=lambda x: (-x["score"], x["candidate_id"]))
for r in top100_sorted[:5]:
    c = candidate_lookup.get(r["candidate_id"])
    if c:
        print(f"\n{r['candidate_id']} | Rank position in list")
        print(generate_reasoning(c, top100_sorted.index(r)+1, r["score"]))


CAND_0079387 | Rank position in list
AI Engineer with 6.9 years experience at Microsoft; verified Sentence Transformers assessment score 86/100; career history shows hands-on retrieval systems, ranking work.

CAND_0018499 | Rank position in list
Senior Machine Learning Engineer with 7.2 years experience at Zomato; verified Deep Learning assessment score 94/100; career history shows hands-on retrieval systems, ranking work.

CAND_0064326 | Rank position in list
Search Engineer with 7.6 years experience at Sarvam AI; verified Milvus assessment score 76/100; career history shows hands-on ranking, embeddings work.

CAND_0041610 | Rank position in list
Recommendation Systems Engineer with 6.7 years experience at Zoho; verified OpenCV assessment score 73/100; career history shows hands-on ranking, embeddings work.

CAND_0027691 | Rank position in list
NLP Engineer with 6.5 years experience at Haptik; verified LoRA assessment score 87/100; career history shows hands-on retrieval systems, ran

In [30]:
import csv, json

top100_sorted = sorted(top500[:100], key=lambda x: (-x["score"], x["candidate_id"]))

with open("submission.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["candidate_id", "rank", "score", "reasoning"])
    
    for i, r in enumerate(top100_sorted, 1):
        c = candidate_lookup.get(r["candidate_id"])
        if c:
            reasoning = generate_reasoning(c, i, r["score"])
        else:
            reasoning = f"{r['title']} with {r['yoe']} years experience. Strong match to Senior AI Engineer role."
        
        writer.writerow([r["candidate_id"], i, round(r["score"], 4), reasoning])

print("submission.csv saved with specific reasoning!")

submission.csv saved with specific reasoning!


In [31]:
rank_py = '''import argparse, gzip, json, heapq, csv
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from datetime import datetime, date

parser = argparse.ArgumentParser()
parser.add_argument("--candidates", required=True)
parser.add_argument("--out", required=True)
args = parser.parse_args()

model = SentenceTransformer("all-MiniLM-L6-v2")

JD_TEXT = """
Senior AI Engineer with production experience in embeddings-based retrieval systems,
vector databases, hybrid search, ranking evaluation metrics like NDCG and MRR,
strong Python, applied ML at product companies, LLM fine-tuning with LoRA or QLoRA,
building recommendation and ranking systems, shipping end-to-end ML systems to real users,
evaluation frameworks for ranking, A/B testing, retrieval quality measurement.
"""
jd_embedding = model.encode([JD_TEXT])

def is_disqualified(c):
    consulting = ["TCS","Infosys","Wipro","Accenture","Cognizant","Capgemini"]
    companies = [r["company"] for r in c["career_history"]]
    if all(any(con in comp for con in consulting) for comp in companies):
        return True
    last_active = datetime.strptime(c["redrob_signals"]["last_active_date"], "%Y-%m-%d").date()
    if (date.today() - last_active).days > 180:
        return True
    return False

def score_candidate(c):
    sig = c["redrob_signals"]
    career_text = " ".join([r["description"] for r in c["career_history"]])
    skills_text = " ".join([s["name"] for s in c["skills"]])
    full_text = career_text + " " + skills_text
    candidate_embedding = model.encode([full_text])
    semantic_score = cosine_similarity(jd_embedding, candidate_embedding)[0][0]
    if semantic_score < 0.28:
        return 0.0
    ML_KEYWORDS = ["retrieval","ranking","recommendation","embeddings",
                   "vector","nlp","fine-tun","transformer","search"]
    ml_hits = sum(1 for kw in ML_KEYWORDS if kw in career_text.lower())
    semantic_score = min(semantic_score + ml_hits * 0.03, 1.0)
    IRRELEVANT_TITLES = ["graphic","civil","mechanical","accountant","marketing",
                         "operations","mobile","project manager","customer support",
                         "business analyst","qa","frontend","java",".net"]
    title = c["profile"]["current_title"].lower()
    title_penalty = 0.25 if any(t in title for t in IRRELEVANT_TITLES) else 1.0
    consulting = ["TCS","Infosys","Wipro","Accenture","Cognizant","Capgemini"]
    exp_score = 0
    yoe = c["profile"]["years_of_experience"]
    if 5 <= yoe <= 9: exp_score += 0.4
    elif 4 <= yoe <= 11: exp_score += 0.2
    for role in c["career_history"]:
        if not any(con in role["company"] for con in consulting):
            exp_score += 0.1
    exp_score = min(exp_score, 1.0)
    sig = c["redrob_signals"]
    avail_score = 0
    if sig["open_to_work_flag"]: avail_score += 0.3
    last_active = datetime.strptime(sig["last_active_date"], "%Y-%m-%d").date()
    days_inactive = (date.today() - last_active).days
    if days_inactive < 30: avail_score += 0.3
    elif days_inactive < 90: avail_score += 0.2
    avail_score += sig["recruiter_response_rate"] * 0.2
    if sig["notice_period_days"] <= 30: avail_score += 0.2
    elif sig["notice_period_days"] <= 60: avail_score += 0.1
    avail_score = min(avail_score, 1.0)
    verify_score = 0
    assessed = sig["skill_assessment_scores"]
    if assessed:
        verify_score += (sum(assessed.values()) / len(assessed) / 100) * 0.5
    if sig["github_activity_score"] > 0:
        verify_score += (sig["github_activity_score"] / 100) * 0.3
    if sig["verified_email"] and sig["verified_phone"]: verify_score += 0.2
    bonus_score = 0
    nice_skills = ["lora","qlora","peft","fine-tuning","open-source",
                   "learning to rank","xgboost","hr-tech","recruiting"]
    candidate_skills_lower = [s["name"].lower() for s in c["skills"]]
    for skill in nice_skills:
        if any(skill in s for s in candidate_skills_lower): bonus_score += 0.15
    bonus_score = min(bonus_score, 1.0)
    short_stints = [r for r in c["career_history"]
                    if r["duration_months"] < 12 and not r["is_current"]]
    hopper_penalty = 1.0
    if len(short_stints) >= 2: hopper_penalty = 0.7
    if len(short_stints) >= 3: hopper_penalty = 0.5
    final = (semantic_score*0.35 + exp_score*0.25 +
             avail_score*0.20 + verify_score*0.12 + bonus_score*0.08)
    return round(final * title_penalty * hopper_penalty, 4)

def generate_reasoning(c, rank):
    sig = c["redrob_signals"]
    profile = c["profile"]
    title = profile["current_title"]
    yoe = profile["years_of_experience"]
    company = profile["current_company"]
    assessed = sig["skill_assessment_scores"]
    career_text = " ".join([r["description"] for r in c["career_history"]]).lower()
    reasoning = f"{title} with {yoe} years experience at {company}"
    if assessed:
        best_skill = max(assessed, key=assessed.get)
        reasoning += f"; verified {best_skill} score {assessed[best_skill]:.0f}/100"
    ml_signals = [kw for kw in ["retrieval","ranking","recommendation","embeddings","nlp"]
                  if kw in career_text]
    if ml_signals:
        reasoning += f"; hands-on {\', \'.join(ml_signals[:2])} work in career history"
    concerns = []
    if sig["notice_period_days"] > 60: concerns.append(f"notice period {sig[\'notice_period_days\']} days")
    if sig["recruiter_response_rate"] < 0.3: concerns.append("low response rate")
    if rank > 50: concerns.append("borderline fit")
    if concerns: reasoning += f". Concern: {\'; \'.join(concerns)}."
    else: reasoning += "."
    return reasoning

TOP_N = 500
top_candidates = []
consulting = ["TCS","Infosys","Wipro","Accenture","Cognizant","Capgemini"]

opener = gzip.open if args.candidates.endswith(".gz") else open
with opener(args.candidates, "rt") as f:
    for line in f:
        if not line.strip(): continue
        c = json.loads(line)
        if is_disqualified(c):
            del c
            continue
        score = score_candidate(c)
        if score > 0:
            if len(top_candidates) < TOP_N:
                heapq.heappush(top_candidates, (score, c["candidate_id"], c))
            elif score > top_candidates[0][0]:
                heapq.heapreplace(top_candidates, (score, c["candidate_id"], c))
        del c

final_ranking = sorted(top_candidates, key=lambda x: (-x[0], x[1]))[:100]

with open(args.out, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["candidate_id", "rank", "score", "reasoning"])
    for i, (score, cid, c) in enumerate(final_ranking, 1):
        writer.writerow([cid, i, score, generate_reasoning(c, i)])

print(f"Saved {args.out}")
'''

with open("rank.py", "w") as f:
    f.write(rank_py)

print("rank.py saved!")

rank.py saved!
